<a href="https://colab.research.google.com/github/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/blob/main/ML_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Session 1: FIND-S Algorithm
import csv
import urllib.request

def find_s_algorithm(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    # Initialize hypothesis with the most specific hypothesis: ['0', '0', ...]
    num_attributes = len(data[0]) - 1
    hypothesis = ['0'] * num_attributes

    print("Initial Hypothesis:", hypothesis)
    print("-" * 50)

    for i, row in enumerate(data):
        features = row[:-1]
        target = row[-1]

        # Consider only positive training instances
        if target.strip().lower() in ['yes', '1', 'positive']:
            if hypothesis == ['0'] * num_attributes:
                # First positive example initializes the hypothesis
                hypothesis = features.copy()
            else:
                for j in range(num_attributes):
                    if hypothesis[j] != features[j]:
                        hypothesis[j] = '?'
            print(f"Instance {i + 1} ({target}): {features}")
            print(f"Hypothesis after instance {i + 1}: {hypothesis}\n")
        else:
            print(f"Instance {i + 1} ({target}): {features} (Ignored)\n")

    print("-" * 50)
    print("Most Specific Hypothesis Found:")
    print(hypothesis)

# Run Find-S using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
find_s_algorithm(github_training_data_url)

Initial Hypothesis: ['0', '0', '0', '0', '0', '0']
--------------------------------------------------
Instance 1 (Yes): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Hypothesis after instance 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']

Instance 2 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']
Hypothesis after instance 2: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Instance 3 (No): ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] (Ignored)

Instance 4 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
Hypothesis after instance 4: ['Sunny', 'Warm', '?', 'Strong', '?', '?']

--------------------------------------------------
Most Specific Hypothesis Found:
['Sunny', 'Warm', '?', 'Strong', '?', '?']


In [ ]:
# Session 2: Candidate-Elimination Algorithm
import csv
import urllib.request

def candidate_elimination(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    num_attributes = len(data[0]) - 1

    # Initialize Specific and General Boundaries
    specific_h = ['0'] * num_attributes
    general_h = [['?' for _ in range(num_attributes)]]

    # Initialize S with the first positive example
    for row in data:
        if row[-1].strip().lower() in ['yes', '1', 'positive']:
            specific_h = row[:-1].copy()
            break

    print("Initial Specific Boundary (S):", specific_h)
    print("Initial General Boundary (G):", general_h)
    print("-" * 60)

    for idx, row in enumerate(data):
        features = row[:-1]
        label = row[-1].strip().lower()

        if label in ['yes', '1', 'positive']:
            # Positive Example
            # Remove any hypothesis from G that is inconsistent with features
            general_h = [
                g for g in general_h
                if all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes))
            ]

            # Generalize S to cover features
            for i in range(num_attributes):
                if specific_h[i] != features[i]:
                    specific_h[i] = '?'

        else:
            # Negative Example
            # Specialize G to exclude features while remaining consistent with S
            new_general_h = []
            for g in general_h:
                if not all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes)):
                    new_general_h.append(g)
                else:
                    for i in range(num_attributes):
                        if g[i] == '?' and specific_h[i] != '?' and specific_h[i] != features[i]:
                            spec = g.copy()
                            spec[i] = specific_h[i]
                            new_general_h.append(spec)

            # Filter duplicates and redundant hypotheses
            general_h = [
                h for i, h in enumerate(new_general_h)
                if h not in new_general_h[:i]
            ]

        print(f"Step {idx + 1} (Instance: {features} -> {row[-1]}):")
        print("  S:", specific_h)
        print("  G:", general_h)
        print("-" * 60)

    print("Final Version Space:")
    print("Specific Boundary (S):", specific_h)
    print("General Boundary (G):", general_h)

# Run Candidate-Elimination using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
candidate_elimination(github_training_data_url)

Initial Specific Boundary (S): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Initial General Boundary (G): [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 1 (Instance: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 2 (Instance: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 3 (Instance: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]
------------------------------------------------------------
Step 4 (Insta

In [ ]:
# Session 3: Decision Tree using ID3 Algorithm
import pandas as pd
import numpy as np

# 1. Calculate Entropy of a dataset
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = 0
    for i in range(len(elements)):
        probability = counts[i] / np.sum(counts)
        entropy_val -= probability * np.log2(probability)
    return entropy_val

# 2. Calculate Information Gain for a feature
def information_gain(data, split_attribute_name, target_attribute_name="PlayTennis"):
    total_entropy = entropy(data[target_attribute_name])

    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = 0

    for i in range(len(vals)):
        sub_data = data[data[split_attribute_name] == vals[i]]
        prob = counts[i] / np.sum(counts)
        weighted_entropy += prob * entropy(sub_data[target_attribute_name])

    info_gain = total_entropy - weighted_entropy
    return info_gain

# 3. ID3 Algorithm Recursive Function
def id3(data, original_data, features, target_attribute_name="PlayTennis", parent_node_class=None):
    # Case 1: If target values are homogeneous (all 'Yes' or all 'No')
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # Case 2: If dataset is empty, return target feature value in original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[
            np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])
        ]

    # Case 3: If feature set is empty, return parent node class
    elif len(features) == 0:
        return parent_node_class

    # Standard Case: Build tree recursively
    else:
        # Determine majority class for current node
        counts = np.unique(data[target_attribute_name], return_counts=True)[1]
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(counts)]

        # Select feature with maximum Information Gain
        item_values = [information_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree root structure
        tree = {best_feature: {}}

        # Remove best feature from feature list
        features = [i for i in features if i != best_feature]

        # Recursively grow tree for each branch/value
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

# 4. Predict function for new unseen sample
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except KeyError:
                return default

            result = tree[key][query[key]]
            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# --- Main Execution ---
if __name__ == "__main__":
    # GitHub Raw URL for play_tennis.csv
    url = "https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/play_tennis.csv"

    # Load dataset directly from URL
    df = pd.read_csv(url)

    # Clean data (Remove 'Day' or similar index columns if present)
    if 'Day' in df.columns:
        df = df.drop(columns=['Day'])
    if 'Humidity' in df.columns:
        df = df[df['Humidity'] != '???']

    # Features & Target Column
    features = [col for col in df.columns if col != 'PlayTennis']

    # Build Decision Tree
    decision_tree = id3(df, df, features)

    print("Constructed Decision Tree Structure:")
    import pprint
    pprint.pprint(decision_tree)
    print("-" * 50)

    # Predict a new sample
    new_sample = {
        'Outlook': 'Sunny',
        'Temperature': 'Cool',
        'Humidity': 'High',
        'Wind': 'Strong'
    }

    prediction = predict(new_sample, decision_tree)
    print("New Sample:", new_sample)
    print("Classification Result for New Sample (PlayTennis):", prediction)

Constructed Decision Tree Structure:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Light': 'Yes', 'Strong': 'No'}},
             'Sunny': {'Temperature': {'Cool': 'Yes',
                                       'Hot': 'No',
                                       'Mild': 'Yes'}}}}
--------------------------------------------------
New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Classification Result for New Sample (PlayTennis): Yes


In [2]:
# Session 4: Artificial Neural Network with Backpropagation

import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([
    [0],
    [1],
    [1],
    [0]
])


np.random.seed(42)


epochs = 10000
learning_rate = 0.1
input_size = 2
hidden_size = 3
output_size = 1


wh = np.random.uniform(size=(input_size, hidden_size))
bh = np.random.uniform(size=(1, hidden_size))


wout = np.random.uniform(size=(hidden_size, output_size))
bout = np.random.uniform(size=(1, output_size))

print("--- Training Progress ---")

for epoch in range(epochs):

    hidden_layer_input = np.dot(X, wh) + bh
    hidden_layer_activations = sigmoid(hidden_layer_input)

    output_layer_input = np.dot(hidden_layer_activations, wout) + bout
    predicted_output = sigmoid(output_layer_input)

    error = y - predicted_output
    d_predicted_output = error * sigmoid_derivative(predicted_output)


    error_hidden_layer = d_predicted_output.dot(wout.T)
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_activations)


    wout += hidden_layer_activations.T.dot(d_predicted_output) * learning_rate
    bout += np.sum(d_predicted_output, axis=0, keepdims=True) * learning_rate

    wh += X.T.dot(d_hidden_layer) * learning_rate
    bh += np.sum(d_hidden_layer, axis=0, keepdims=True) * learning_rate


    if (epoch + 1) % 2000 == 0 or epoch == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {loss:.6f}")

print("-" * 50)
print("Training Completed.\n")


print("--- Testing Results ---")
print("Input 1 | Input 2 | Target Output | Predicted Output | Classification")
print("-" * 68)
for i in range(len(X)):

    hidden_act = sigmoid(np.dot(X[i], wh) + bh)
    pred = sigmoid(np.dot(hidden_act, wout) + bout)[0][0]
    binary_class = 1 if pred > 0.5 else 0
    print(f"   {X[i][0]}    |    {X[i][1]}    |       {y[i][0]}       |     {pred:.4f}     |       {binary_class}")

--- Training Progress ---
Epoch 1/10000 - Loss (MSE): 0.394174
Epoch 2000/10000 - Loss (MSE): 0.221589
Epoch 4000/10000 - Loss (MSE): 0.142439
Epoch 6000/10000 - Loss (MSE): 0.128289
Epoch 8000/10000 - Loss (MSE): 0.018662
Epoch 10000/10000 - Loss (MSE): 0.007339
--------------------------------------------------
Training Completed.

--- Testing Results ---
Input 1 | Input 2 | Target Output | Predicted Output | Classification
--------------------------------------------------------------------
   0    |    0    |       0       |     0.0501     |       0
   0    |    1    |       1       |     0.9208     |       1
   1    |    0    |       1       |     0.9036     |       1
   1    |    1    |       0       |     0.1061     |       0


In [5]:
# Session 5 : Naïve Bayes Algorithm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="dataset_session5.csv"):
    np.random.seed(42)
    n_samples = 200

    age = np.random.randint(18, 65, size=n_samples)
    salary = np.random.randint(20000, 120000, size=n_samples)
    credit_score = np.random.randint(300, 850, size=n_samples)

    purchased = ((age * 0.4 + (salary / 2000) * 0.6) > 35).astype(int)

    df = pd.DataFrame({
        'Age': age,
        'EstimatedSalary': salary,
        'CreditScore': credit_score,
        'Purchased': purchased
    })


    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_naive_bayes(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Age', 'EstimatedSalary', 'CreditScore']]
    y = df['Purchased']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    nb_model = GaussianNB()

    nb_model.fit(X_train, y_train)

    y_pred = nb_model.predict(X_test)


    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nMatrix Structure:")
    print(f"  [ True Negatives (TN): {cm[0][0]} | False Positives (FP): {cm[0][1]} ]")
    print(f"  [ False Negatives (FN): {cm[1][0]} | True Positives (TP):  {cm[1][1]} ]")

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("dataset_session5.csv")
    run_naive_bayes(dataset_file)

Dataset successfully created and saved as 'dataset_session5.csv'.

--- Dataset Preview ---
   Age  EstimatedSalary  CreditScore  Purchased
0   56            72733          335          1
1   46            85318          319          1
2   32           109474          620          1
3   60            43664          811          1
4   25            87172          699          1 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[18  6]
 [ 0 16]]

Matrix Structure:
  [ True Negatives (TN): 18 | False Positives (FP): 6 ]
  [ False Negatives (FN): 0 | True Positives (TP):  16 ]

Accuracy Score: 85.00%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.75      0.86        24
           1       0.73      1.00      0.84        16

    accuracy                           0.85        40
   macro avg       0.86      0.88      0.85    

In [6]:
# Session 6: House Price Prediction using Linear Regression
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


def generate_dataset(filename="house_prices.csv"):
    np.random.seed(42)
    n_samples = 250

    square_feet = np.random.randint(600, 3500, size=n_samples)
    bedrooms = np.random.randint(1, 6, size=n_samples)
    age = np.random.randint(1, 30, size=n_samples)
    distance_to_center = np.random.uniform(1.0, 25.0, size=n_samples).round(1)

    base_price = (
        square_feet * 150 +
        bedrooms * 10000 -
        age * 1200 -
        distance_to_center * 2500 +
        30000
    )
    noise = np.random.normal(0, 15000, size=n_samples)
    price = (base_price + noise).round(2)

    df = pd.DataFrame({
        'SquareFeet': square_feet,
        'Bedrooms': bedrooms,
        'Age': age,
        'DistanceToCenter': distance_to_center,
        'Price': price
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_house_price_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['SquareFeet', 'Bedrooms', 'Age', 'DistanceToCenter']]
    y = df['Price']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    model = LinearRegression()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%\n")

    print("--- Feature Coefficients ---")
    for feature, coef in zip(X.columns, model.coef_):
        print(f"  {feature:18s}: ${coef:,.2f}")
    print(f"  Intercept         : ${model.intercept_:,.2f}\n")

    sample_house = pd.DataFrame({
        'SquareFeet': [2000],
        'Bedrooms': [3],
        'Age': [5],
        'DistanceToCenter': [8.5]
    })
    predicted_price = model.predict(sample_house)[0]

    print("--- Sample Prediction ---")
    print(f"Features: 2000 sqft, 3 Bed, 5 years old, 8.5 km from center")
    print(f"Predicted Price: ${predicted_price:,.2f}")

if __name__ == "__main__":
    dataset_file = generate_dataset("house_prices.csv")
    run_house_price_prediction(dataset_file)

Dataset successfully created and saved as 'house_prices.csv'.

--- Dataset Preview ---
   SquareFeet  Bedrooms  Age  DistanceToCenter      Price
0        1460         2   24              19.0  181008.61
1        1894         2    4              10.4  270352.28
2        1730         3   25              20.9  238077.34
3        1695         4   21              14.7  299885.34
4        2238         2   29               2.5  329370.71 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Mean Absolute Error (MAE) : $14,346.19
Root Mean Squared Error (RMSE): $17,118.90
R² Score (Accuracy)       : 97.89%

--- Feature Coefficients ---
  SquareFeet        : $149.22
  Bedrooms          : $10,274.56
  Age               : $-1,019.83
  DistanceToCenter  : $-2,345.91
  Intercept         : $27,573.22

--- Sample Prediction ---
Features: 2000 sqft, 3 Bed, 5 years old, 8.5 km from center
Predicted Price: $331,801.17


In [7]:
# Session 7: Iris Flower Classification using Naïve Bayes Classifier

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="iris_dataset.csv"):
    iris = load_iris()

    df = pd.DataFrame(data=iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    df['species'] = iris.target_names[iris.target]

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_iris_classification(csv_file):

    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
    y = df['species']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    nb_model = GaussianNB()

    nb_model.fit(X_train, y_train)

    y_pred = nb_model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("iris_dataset.csv")
    run_iris_classification(dataset_file)

Dataset successfully created and saved as 'iris_dataset.csv'.

--- Dataset Preview ---
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Accuracy Score: 100.00%

Detailed Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg  

In [8]:
# Session 8: Implementation of Naïve Bayes Classification for Bank Loan Prediction

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="bank_loan_dataset.csv"):
    np.random.seed(42)
    n_samples = 300

    # Features: Income (in k$), Credit Score, Debt-to-Income Ratio (%), Loan Amount (in k$)
    income = np.random.randint(20, 150, size=n_samples)
    credit_score = np.random.randint(300, 850, size=n_samples)
    dti_ratio = np.random.uniform(5.0, 50.0, size=n_samples).round(1)
    loan_amount = np.random.randint(5, 50, size=n_samples)

    approval_score = (income * 0.4) + (credit_score * 0.08) - (dti_ratio * 0.8) - (loan_amount * 0.5)
    loan_approved = (approval_score > 45).astype(int)

    df = pd.DataFrame({
        'Income': income,
        'CreditScore': credit_score,
        'DTIRatio': dti_ratio,
        'LoanAmount': loan_amount,
        'LoanApproved': loan_approved
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_bank_loan_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Income', 'CreditScore', 'DTIRatio', 'LoanAmount']]
    y = df['LoanApproved']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )


    nb_model = GaussianNB()


    nb_model.fit(X_train, y_train)


    y_pred = nb_model.predict(X_test)


    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nMatrix Structure:")
    print(f"  [ True Negatives (TN): {cm[0][0]} | False Positives (FP): {cm[0][1]} ]")
    print(f"  [ False Negatives (FN): {cm[1][0]} | True Positives (TP):  {cm[1][1]} ]")

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("bank_loan_dataset.csv")
    run_bank_loan_prediction(dataset_file)

Dataset successfully created and saved as 'bank_loan_dataset.csv'.

--- Dataset Preview ---
   Income  CreditScore  DTIRatio  LoanAmount  LoanApproved
0     122          405      10.8          12             1
1     112          419      10.8          38             1
2      34          357      11.8          39             0
3     126          645      11.2          36             1
4      91          773      33.8          28             1 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[25  5]
 [ 0 30]]

Matrix Structure:
  [ True Negatives (TN): 25 | False Positives (FP): 5 ]
  [ False Negatives (FN): 0 | True Positives (TP):  30 ]

Accuracy Score: 91.67%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91        30
           1       0.86      1.00      0.92        30

    accuracy                      